# Notebook 3 — ML Model

XGBoost with SHAP explainability + walk-forward backtesting.

**Features**: 31 features — fraud signals (numeric + binary flags) + value metrics from each annual 10-K  
**Labels**: four targets trained separately:
- `forward_return_12m` — regression (predict exact 12m return)
- `beat_sp500` — binary classification (beat S&P 500 over 12m?)
- `forward_return_24m` — regression (predict exact 24m return)
- `beat_sp500_24m` — binary classification (beat S&P 500 over 24m?)

**Validation strategy**: walk-forward splits — never train on future data:
```
Split 1: train 2021-2022 → test 2023
Split 2: train 2021-2023 → test 2024
Split 3: train 2021-2024 → test 2025
```

## Install dependencies (first time only)

In [ ]:
# Uncomment to install
# !pip install xgboost shap scikit-learn
# Mac users: if xgboost fails with libomp error → brew install libomp

## Load data

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')

import xgboost as xgb
import shap
from sklearn.metrics import roc_auc_score, mean_absolute_error

print(f"XGBoost version: {xgb.__version__}")

# ── Load dataset ──────────────────────────────────────────────────────────────
IN_COLAB = 'google.colab' in str(get_ipython())
if IN_COLAB:
    import subprocess
    subprocess.run(['wget', '-q', '-O', 'historical_dataset.parquet',
        'https://raw.githubusercontent.com/sherlock718/stock-fraud-screener/main/data/historical_dataset.parquet'])
    path = 'historical_dataset.parquet'
else:
    path = '../data/historical_dataset.parquet'

df = pd.read_parquet(path)
# Filter out malformed fiscal years
df = df[df['fiscal_year'].between(2000, 2030)]
print(f"Rows: {len(df):,} | Companies: {df['ticker'].nunique():,}")
print(f"12m forward returns: {df['forward_return_12m'].notna().sum():,}")
print(f"24m forward returns: {df['forward_return_24m'].notna().sum():,} (older filings only)")

## Feature engineering

In [ ]:
FEATURES = [
    # Fraud signals (numeric)
    'beneish_score', 'piotroski_score', 'accruals_ratio', 'cfd_ratio',
    'altman_score', 'ar_ratio', 'dso', 'non_op_ratio',
    # Binary fraud flags (0/1)
    'beneish_flag', 'piotroski_weak', 'accruals_flag', 'cfd_flag',
    'altman_flag', 'revenue_quality_flag', 'earnings_quality_flag',
    # Value metrics
    'earnings_yield', 'return_on_capital', 'acquirers_multiple',
    'gross_profitability', 'croic', 'roe', 'roa', 'fcf_yield',
    'gross_margin', 'net_margin', 'debt_to_equity', 'current_ratio',
    'pe_ratio', 'pb_ratio', 'ev_ebitda', 'ncav_ratio',
]

def prep_dataset(label_col):
    """Prepare X, y, years for a given label column."""
    sub = df[df[label_col].notna()].copy()
    sub[label_col] = sub[label_col].clip(
        sub[label_col].quantile(0.01), sub[label_col].quantile(0.99)
    )
    feats = [f for f in FEATURES if f in sub.columns]
    X = sub[feats].copy()
    for col in X.select_dtypes(include=[float, int]).columns:
        lo, hi = X[col].quantile(0.01), X[col].quantile(0.99)
        X[col] = X[col].clip(lo, hi)
    return X, sub[label_col], sub['fiscal_year'], sub, feats

X_12, y_12, years_12, ml_df_12, features_12 = prep_dataset('forward_return_12m')
X_24, y_24, years_24, ml_df_24, features_24 = prep_dataset('forward_return_24m')

beat_12 = ml_df_12['beat_sp500'].astype(float)
beat_24 = ml_df_24['beat_sp500_24m'].astype(float)

print(f"12m dataset: {X_12.shape[0]:,} rows × {X_12.shape[1]} features | label balance: {beat_12.mean():.1%}")
print(f"24m dataset: {X_24.shape[0]:,} rows × {X_24.shape[1]} features | label balance: {beat_24.mean():.1%}")

## Walk-forward validation — 12m horizon

In [ ]:
def run_walk_forward(X, y_reg, y_cls, years, features, label='12m'):
    test_years = sorted(years.unique())[2:]
    print(f"Walk-forward test years ({label}): {test_years}")

    all_preds_cls, all_true_cls = [], []
    all_preds_reg, all_true_reg = [], []
    all_test_years = []
    feature_importances = []

    for test_year in test_years:
        train_mask = years < test_year
        test_mask  = years == test_year
        if train_mask.sum() < 50 or test_mask.sum() < 10:
            print(f"  {test_year}: skipped")
            continue

        X_tr, X_te = X[train_mask], X[test_mask]
        y_tr_cls = y_cls[train_mask]; y_te_cls = y_cls[test_mask]
        y_tr_reg = y_reg[train_mask]; y_te_reg = y_reg[test_mask]

        clf = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                                  subsample=0.8, colsample_bytree=0.8,
                                  eval_metric='logloss', random_state=42, n_jobs=-1)
        clf.fit(X_tr.fillna(X_tr.median()), y_tr_cls.fillna(0))
        pred_proba = clf.predict_proba(X_te.fillna(X_tr.median()))[:, 1]

        reg = xgb.XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                                 subsample=0.8, colsample_bytree=0.8,
                                 random_state=42, n_jobs=-1)
        reg.fit(X_tr.fillna(X_tr.median()), y_tr_reg.fillna(0))
        pred_reg = reg.predict(X_te.fillna(X_tr.median()))

        all_preds_cls.extend(pred_proba); all_true_cls.extend(y_te_cls.values)
        all_preds_reg.extend(pred_reg);   all_true_reg.extend(y_te_reg.values)
        all_test_years.extend([test_year] * len(y_te_cls))
        feature_importances.append(pd.Series(clf.feature_importances_, index=features))

        auc = roc_auc_score(y_te_cls.fillna(0), pred_proba)
        mae = mean_absolute_error(y_te_reg, pred_reg)
        print(f"  {test_year}: train={train_mask.sum():,} test={test_mask.sum():,} | AUC={auc:.3f} | MAE={mae:.3f}")

    if all_preds_cls:
        print(f"\n{label} Overall AUC: {roc_auc_score(all_true_cls, all_preds_cls):.3f}")
        print(f"{label} Overall MAE: {mean_absolute_error(all_true_reg, all_preds_reg):.3f}")
    else:
        print("Not enough data for walk-forward yet.")

    return all_preds_cls, all_true_cls, all_preds_reg, all_true_reg, all_test_years, feature_importances

res12 = run_walk_forward(X_12, y_12, beat_12, years_12, features_12, '12m')

## Walk-forward validation — 24m horizon

Fraud signals may take longer to show up in prices — comparing 12m vs 24m AUC tells you which horizon the signals predict better.

In [ ]:
res24 = run_walk_forward(X_24, y_24, beat_24, years_24, features_24, '24m')

## Feature importance — 12m vs 24m comparison

In [ ]:
fi_12 = res12[5]; fi_24 = res24[5]

if fi_12 and fi_24:
    avg_12 = pd.concat(fi_12, axis=1).mean(axis=1).sort_values(ascending=False)
    avg_24 = pd.concat(fi_24, axis=1).mean(axis=1).sort_values(ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    for ax, avg, title in [(axes[0], avg_12, '12m Horizon'), (axes[1], avg_24, '24m Horizon')]:
        top15 = avg.head(15)
        ax.barh(top15.index[::-1], top15.values[::-1], color='#3498db', alpha=0.85)
        ax.set_title(f'Top 15 Features — {title}')
        ax.set_xlabel('XGBoost Feature Importance (gain)')
    plt.suptitle('Which signals matter at 12m vs 24m?', y=1.02)
    plt.tight_layout()
    plt.show()
elif fi_12:
    avg_12 = pd.concat(fi_12, axis=1).mean(axis=1).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(10, 7))
    top20 = avg_12.head(20)
    ax.barh(top20.index[::-1], top20.values[::-1], color='#3498db', alpha=0.85)
    ax.set_title('Top 20 Features — 12m Horizon')
    plt.tight_layout(); plt.show()

## SHAP — explainable predictions (12m model)

In [ ]:
X_all = X_12.fillna(X_12.median())
y_all = beat_12.fillna(0)

final_clf = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=42, n_jobs=-1
)
final_clf.fit(X_all, y_all)

explainer = shap.TreeExplainer(final_clf)
shap_values = explainer.shap_values(X_all)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_all, plot_type='bar', show=False)
plt.title('SHAP Feature Importance — Beat S&P 500 (12m)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_all, show=False)
plt.title('SHAP Beeswarm — How Features Push Predictions (12m)')
plt.tight_layout()
plt.show()

## Backtest equity curve

Each year: go long the top-20 stocks by ML predicted probability of beating S&P 500 (12m model).  
Compare cumulative returns vs holding SPY.

In [ ]:
TOP_N = 20

preds_cls_12, _, _, _, test_yrs_12, _ = res12

# Rebuild predictions using walk-forward for backtest
test_years_12 = sorted(years_12.unique())[2:]
pred_series = pd.Series(np.nan, index=ml_df_12.index)
for test_year in test_years_12:
    train_mask = years_12 < test_year
    test_mask  = years_12 == test_year
    if train_mask.sum() < 50 or test_mask.sum() < 10:
        continue
    clf2 = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                               subsample=0.8, colsample_bytree=0.8,
                               eval_metric='logloss', random_state=42, n_jobs=-1)
    clf2.fit(X_12[train_mask].fillna(X_12[train_mask].median()), beat_12[train_mask].fillna(0))
    preds = clf2.predict_proba(X_12[test_mask].fillna(X_12[train_mask].median()))[:, 1]
    pred_series.loc[X_12[test_mask].index] = preds

backtest_df = ml_df_12.copy()
backtest_df['pred_proba'] = pred_series
backtest_df = backtest_df[backtest_df['pred_proba'].notna()]

if len(backtest_df) < 10:
    print("Not enough data for backtest yet.")
else:
    results = []
    for yr, grp in backtest_df.groupby('fiscal_year'):
        top = grp.nlargest(TOP_N, 'pred_proba')
        results.append({
            'year': yr,
            'strategy': top['forward_return_12m'].mean(),
            'sp500': top['sp500_return_12m'].mean() if 'sp500_return_12m' in top.columns else None,
            'n_stocks': len(top)
        })

    res_df = pd.DataFrame(results).dropna(subset=['strategy'])
    res_df['cum_strategy'] = (1 + res_df['strategy']).cumprod()
    res_df['cum_sp500']    = (1 + res_df['sp500'].fillna(0)).cumprod()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    x = np.arange(len(res_df)); w = 0.35
    axes[0].bar(x - w/2, res_df['strategy'], w, label=f'Strategy (Top {TOP_N})', color='#2ecc71', alpha=0.85)
    axes[0].bar(x + w/2, res_df['sp500'],    w, label='S&P 500', color='#3498db', alpha=0.85)
    axes[0].set_xticks(x); axes[0].set_xticklabels(res_df['year'].astype(int))
    axes[0].axhline(0, color='white', alpha=0.3)
    axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    axes[0].set_title('Annual Returns — Strategy vs S&P 500'); axes[0].legend()

    axes[1].plot(res_df['year'], res_df['cum_strategy'], marker='o', color='#2ecc71', label='Strategy')
    axes[1].plot(res_df['year'], res_df['cum_sp500'],    marker='o', color='#3498db', label='S&P 500')
    axes[1].set_title('Cumulative Return')
    axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    axes[1].legend()

    plt.suptitle(f'Backtest: Top {TOP_N} stocks by ML score each year (12m)', y=1.02)
    plt.tight_layout(); plt.show()

    print(res_df[['year', 'strategy', 'sp500', 'n_stocks']].to_string(index=False))
    print(f"\nFinal strategy return: {res_df['cum_strategy'].iloc[-1]-1:.1%}")
    print(f"Final S&P 500 return:  {res_df['cum_sp500'].iloc[-1]-1:.1%}")

## Inspect individual predictions

SHAP waterfall for any company — exactly why the model gave that prediction.

In [ ]:
INSPECT_TICKER = 'AAPL'

rows = X_all[ml_df_12['ticker'] == INSPECT_TICKER]
if len(rows) == 0:
    print(f"No data found for {INSPECT_TICKER}")
else:
    row_idx = rows.index[-1]
    row = rows.loc[[row_idx]]
    year = ml_df_12.loc[row_idx, 'fiscal_year']
    actual = ml_df_12.loc[row_idx, 'forward_return_12m']
    pred = final_clf.predict_proba(row.fillna(X_all.median()))[0, 1]
    print(f"{INSPECT_TICKER} — FY{year} | Actual 12m: {actual:.1%} | Beat S&P prob: {pred:.1%}")

    shap_row = explainer.shap_values(row)
    shap.waterfall_plot(
        shap.Explanation(values=shap_row[0],
                         base_values=explainer.expected_value,
                         data=row.values[0],
                         feature_names=features_12),
        show=True
    )